# 04 · Business impact: from risk scores to decisions

**Question:** how can a hotel use the cancellation risk in practice, and what could it be worth?

**Data:** the 40,619 bookings arriving in 2017, with the risk score of the final model (notebook 03). The model never saw these bookings during training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/test_predictions.csv", parse_dates=["arrival_date"])
total_cancels = df["is_canceled"].sum()
total_lost = df.loc[df["is_canceled"] == 1, "revenue"].sum()

print(df.shape)
print(f"Actual cancellation rate: {df['is_canceled'].mean():.1%}")
print(f"Average predicted risk:   {df['cancel_score'].mean():.1%}")

## 4.1 Calibration: can the probabilities be trusted?
Bookings are split into 10 equal groups by predicted risk. For each group, the average predicted risk is compared with the actual cancellation rate. On the diagonal, "40% risk" means that 40% of those bookings are really cancelled.

In [ ]:
df["risk_decile"] = pd.qcut(df["cancel_score"], 10, labels=False, duplicates="drop") + 1
calib = df.groupby("risk_decile").agg(
    n_bookings=("is_canceled", "size"),
    predicted=("cancel_score", "mean"),
    actual=("is_canceled", "mean"),
)
calib[["predicted", "actual"]] = (calib[["predicted", "actual"]] * 100).round(1)
calib["gap"] = (calib["actual"] - calib["predicted"]).round(1)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 100], [0, 100], color="#bbb", linewidth=1, linestyle="--", label="Perfect calibration")
ax.plot(calib["predicted"], calib["actual"], color="#2a78d6", linewidth=2, marker="o", markersize=6, label="Model")
ax.set_xlabel("Predicted cancellation risk (%)")
ax.set_ylabel("Actual cancellation rate (%)")
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
ax.grid(color="#eee", linewidth=0.8)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.set_title("Calibration on 2017 bookings", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/calibration.png", dpi=150, bbox_inches="tight")
plt.show()
calib

The `gap` column shows, for each group, how far the actual rate is from the predicted one (positive = the model underestimates the risk). Groups close to the diagonal can be used directly for planning; systematic gaps would call for recalibration before use.

## 4.2 Choosing the threshold
For each threshold: bookings flagged, precision, recall, guests flagged by mistake and share of the revenue of cancelled bookings that is covered.

In [ ]:
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    flagged = df["cancel_score"] >= t
    hits = flagged & (df["is_canceled"] == 1)
    rows.append({
        "threshold": t,
        "flagged": int(flagged.sum()),
        "pct_flagged": round(flagged.mean() * 100, 1),
        "precision": round(hits.sum() / flagged.sum() * 100, 1),
        "recall": round(hits.sum() / total_cancels * 100, 1),
        "false_alarms": int((flagged & (df["is_canceled"] == 0)).sum()),
        "revenue_covered_pct": round(df.loc[hits, "revenue"].sum() / total_lost * 100, 1),
    })
thresholds = pd.DataFrame(rows)
thresholds

Lowering the threshold finds more cancellations at the cost of more false alarms. The right choice depends on the action: a reminder email is cheap, so a low threshold is acceptable; a deposit request or overbooking is costly if wrong, so it needs a high threshold. This is why three tiers are used instead of a single cut-off.

## 4.3 Risk tiers and actions

| Tier | Risk | Action |
|---|---|---|
| Low | < 30% | No action |
| Medium | 30–60% | Reminder email with easy reconfirmation |
| High | ≥ 60% | Reconfirmation call or deposit request; include in overbooking |

In [ ]:
df["risk_tier"] = pd.cut(df["cancel_score"], bins=[0, 0.3, 0.6, 1.0],
                         labels=["1. Low", "2. Medium", "3. High"], include_lowest=True)
df["revenue_if_cancelled"] = df["revenue"].where(df["is_canceled"] == 1, 0)

tiers = df.groupby("risk_tier", observed=True).agg(
    n_bookings=("is_canceled", "size"),
    cancel_rate=("is_canceled", "mean"),
    revenue_lost=("revenue_if_cancelled", "sum"),
)
tiers["pct_bookings"] = (tiers["n_bookings"] / len(df) * 100).round(1)
tiers["cancel_rate"] = (tiers["cancel_rate"] * 100).round(1)
tiers["pct_revenue_lost"] = (tiers["revenue_lost"] / total_lost * 100).round(1)
tiers["revenue_lost"] = tiers["revenue_lost"].round(0)
tiers

## 4.4 Expected vs actual cancellations per week
Summing the risk scores of all bookings arriving in a week gives the **expected** number of cancellations for that week: the figure a revenue manager needs to plan overbooking. Individual errors partly cancel out at this level.

In [ ]:
weekly = (df.set_index("arrival_date")
            .resample("W")
            .agg({"cancel_score": "sum", "is_canceled": "sum"})
            .rename(columns={"cancel_score": "expected", "is_canceled": "actual"}))
weekly = weekly.iloc[1:-1]  # first and last week are only partly inside the test period

weekly_error = (weekly["expected"] - weekly["actual"]).abs().sum() / weekly["actual"].sum()
print(f"Weeks: {len(weekly)} | expected: {weekly['expected'].sum():,.0f} | "
      f"actual: {weekly['actual'].sum():,.0f} | average weekly error: {weekly_error:.1%}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(weekly.index, weekly["actual"], color="#2a78d6", linewidth=2, label="Actual cancellations")
ax.plot(weekly.index, weekly["expected"], color="#eb6834", linewidth=2, linestyle="--", label="Expected (model)")
ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#ccc")
ax.tick_params(axis="y", length=0, labelsize=9)
ax.tick_params(axis="x", labelsize=9)
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.set_title("Cancellations per arrival week, 2017: expected vs actual", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/expected_vs_actual_weekly.png", dpi=150, bbox_inches="tight")
plt.show()

## 4.5 What could it be worth? (scenario)
**Assumption:** acting early on high-risk bookings (reconfirmation, deposit, overbooking) lets the hotel resell a share of the rooms that would otherwise stay empty. The share is an assumption, not a measured result, so three scenarios are shown.

In [ ]:
high_lost = df.loc[(df["risk_tier"] == "3. High") & (df["is_canceled"] == 1), "revenue"].sum()
scenarios = pd.DataFrame({
    "resold_share": ["10%", "20%", "30%"],
    "revenue_recovered": [round(high_lost * s) for s in [0.10, 0.20, 0.30]],
})
print(f"Revenue of cancelled high-risk bookings (Jan-Aug 2017): {high_lost:,.0f}")
scenarios

## Summary

In [ ]:
t50 = thresholds.set_index("threshold").loc[0.5]
high = tiers.loc["3. High"]
print(f"Calibration: largest gap between predicted and actual rate across deciles: "
      f"{calib['gap'].abs().max():.1f} points")
print(f"Threshold 0.5: {t50['pct_flagged']:.1f}% of bookings flagged, precision {t50['precision']:.1f}%, "
      f"recall {t50['recall']:.1f}%")
print(f"High-risk tier: {high['pct_bookings']:.1f}% of bookings, cancellation rate {high['cancel_rate']:.1f}%, "
      f"{high['pct_revenue_lost']:.1f}% of revenue lost")
print(f"Weekly expected vs actual cancellations: average error {weekly_error:.1%}")
print(f"Scenario: revenue recovered by reselling 10-30% of high-risk cancellations: "
      f"{high_lost * 0.10:,.0f} - {high_lost * 0.30:,.0f}")

## Recommendations

1. **Use risk tiers, not a single yes/no prediction.** Cheap actions (reminder emails) for medium risk; reconfirmation calls, deposit requests and overbooking only for high risk, where the model is most precise.
2. **Plan overbooking on expected cancellations per date,** where individual errors partly cancel out, rather than on single bookings.
3. **Never cancel or reject a booking based on the score alone:** the model ranks risk, it does not know the guest's intentions.

**Caveats:** the test period covers January–August 2017 only; the value of early action is a scenario based on assumptions; revenue is an upper bound (resold rooms and kept deposits are not observed).